# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/notebooks/02_your_first_readable_model.ipynb)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.950
Hand rule  Precision@50: 0.660


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.950   vs   tree 0.650
Precision@50:  hand rule 0.660   vs   tree 0.640


Look closely: the tree **wins at Precision@50** but your hand rule **wins at Precision@20**. Both results are real. A sharp human rule can be excellent at the very top of the list; the model's advantage shows up deeper, where simple rules run out of signal. Saying exactly that — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

In [7]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- impressions_90d <= 15961.50
|   |   |   |--- impressions_90d <= 1333.00
|   |   |   |   |--- class: 0
|   |   |   |--- impressions_90d >  1333.00
|   |   |   |   |--- class: 0
|   |   |--- impressions_90d >  15961.50
|   |   |   |--- days_since_last_update <= 63.00
|   |   |   |   |--- class: 0
|   |   |   |--- days_since_last_update >  63.00
|   |   |   |   |--- class: 1
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



### A note before we start
The stub cell above (`5e35d2e0`) copy-pasted the *leaky* `X_leaky` block from Section 3 — it still feeds `trend_pct` in, which is the answer in disguise. That's exactly the trap the notebook just warned about, so we won't build on it. Everything below uses the honest `features` list from Section 2 (`content_age_days`, `days_since_last_update`, `impressions_90d`, `avg_position`, `ctr`, `word_count`).

#### 1. Does `max_depth` = 3 or 4 improve Precision@50? Is it still readable?

In [8]:
for depth in (2, 3, 4):
    t = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42).fit(X, y)
    p20 = precision_at_k(t.predict_proba(X)[:, 1], y, 20)
    p50 = precision_at_k(t.predict_proba(X)[:, 1], y, 50)
    print(f"depth={depth}  Precision@20={p20:.3f}  Precision@50={p50:.3f}  leaves={t.get_n_leaves()}")

print()
tree_d4 = DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42).fit(X, y)
print(export_text(tree_d4, feature_names=features))

depth=2  Precision@20=0.650  Precision@50=0.640  leaves=4
depth=3  Precision@20=0.650  Precision@50=0.660  leaves=8


depth=4  Precision@20=0.700  Precision@50=0.720  leaves=16



|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- impressions_90d <= 3.50
|   |   |   |--- word_count <= 687.00
|   |   |   |   |--- class: 0
|   |   |   |--- word_count >  687.00
|   |   |   |   |--- class: 0
|   |   |--- impressions_90d >  3.50
|   |   |   |--- content_age_days <= 237.50
|   |   |   |   |--- class: 0
|   |   |   |--- content_age_days >  237.50
|   |   |   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- content_age_days <= 108.50
|   |   |   |--- days_since_last_update <= 14.00
|   |   |   |   |--- class: 1
|   |   |   |--- days_since_last_update >  14.00
|   |   |   |   |--- class: 0
|   |   |--- content_age_days >  108.50
|   |   |   |--- impressions_90d <= 2.50
|   |   |   |   |--- class: 0
|   |   |   |--- impressions_90d >  2.50
|   |   |   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- ctr <= 0.33
|   |   |   |--- impressions_90d <= 73.50
|   |   |   |   |--- class: 1
|   |   |  

**What happened:** Precision@50 does go up as depth increases (roughly 0.64 → 0.66 → 0.72 in this run — exact numbers can shift a hair with your sklearn version, but the trend holds). More splits let the tree carve out smaller, purer pockets of the top-K.

**Can I still read it?** At depth 3 (7 questions max), yes — I can trace a page through it in a few seconds. At depth 4 (15 questions max) it's technically printable but I'm no longer doing it in my head; I'm scanning line by line. That's the real tradeoff: depth buys you precision but spends your ability to *sanity-check the model in 5 seconds*, which was the whole point of Section 2. A depth-2 or depth-3 tree is a rule you can defend in a meeting. A depth-4+ tree is closer to a black box that happens to print as text.

#### 2. Swap in different features — drop `impressions_90d`, add `engagement_rate`. What splits first?

In [9]:
features_swap = ["content_age_days", "days_since_last_update", "engagement_rate",
                 "avg_position", "ctr", "word_count"]
X_swap = df[features_swap].replace([np.inf, -np.inf], np.nan).fillna(0)

tree_swap = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_swap, y)
print(export_text(tree_swap, feature_names=features_swap))

p20 = precision_at_k(tree_swap.predict_proba(X_swap)[:, 1], y, 20)
p50 = precision_at_k(tree_swap.predict_proba(X_swap)[:, 1], y, 50)
print(f"Precision@20={p20:.3f}  Precision@50={p50:.3f}")

for name, imp in zip(features_swap, tree_swap.feature_importances_):
    print(f"{name:24s} {imp:.3f}")

|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- class: 0

Precision@20=0.700  Precision@50=0.660
content_age_days         0.397
days_since_last_update   0.000
engagement_rate          0.000
avg_position             0.603
ctr                      0.000
word_count               0.000


**What happened:** with `impressions_90d` gone, the root split moves to `avg_position` — the tree's next-best proxy for "is this page actually getting seen." `content_age_days` shows up as the second split, same as before. `engagement_rate` doesn't get chosen at all at depth 2 — it has plenty of zeros (see its distribution: median is 0) and, one level deep, `avg_position` and `content_age_days` split the label better. It might still earn a spot deeper in the tree, or interact with other features in a way a depth-2 stump can't see — but on its own it's not the strongest single signal here. That's a useful, non-obvious finding: *the feature you expected to matter isn't automatically the one the tree picks.*

#### 3. The real check — client-holdout, not in-sample
Everything above scored the tree on the same rows it was trained on. `scripts/03_train_model.py` never does that — it splits by `client_id` so a client's pages appear in either train or test, never both. Let's do the same split here by hand and re-run the depth comparison.

In [10]:
RANDOM_STATE = 42

client_series = df["client_id"].fillna("unknown").astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()

rng = np.random.default_rng(RANDOM_STATE)
shuffled_clients = rng.permutation(unique_clients)
test_client_count = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:test_client_count])

test_mask = client_series.isin(test_clients).to_numpy()
train_idx = np.arange(len(df))[~test_mask]
test_idx = np.arange(len(df))[test_mask]

print(f"{len(unique_clients)} clients total | {len(train_idx)} train rows | {len(test_idx)} held-out rows "
      f"from {len(test_clients)} clients never seen in training")

32 clients total | 27675 train rows | 2325 held-out rows from 6 clients never seen in training


In [11]:
print(f"{'':8s} {'train P@50':>12s}   {'held-out P@20':>14s} {'held-out P@50':>14s}")
for depth in (2, 3, 4):
    t = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=RANDOM_STATE)
    t.fit(X.iloc[train_idx], y[train_idx])

    train_p50 = precision_at_k(t.predict_proba(X.iloc[train_idx])[:, 1], y[train_idx], 50)
    test_p20 = precision_at_k(t.predict_proba(X.iloc[test_idx])[:, 1], y[test_idx], 20)
    test_p50 = precision_at_k(t.predict_proba(X.iloc[test_idx])[:, 1], y[test_idx], 50)
    print(f"depth={depth}  {train_p50:>10.3f}   {test_p20:>14.3f} {test_p50:>14.3f}")

hr_test20 = precision_at_k(df["hand_rule_score"].values[test_idx], y[test_idx], 20)
hr_test50 = precision_at_k(df["hand_rule_score"].values[test_idx], y[test_idx], 50)
print(f"\nhand rule on held-out clients:  Precision@20={hr_test20:.3f}  Precision@50={hr_test50:.3f}")

           train P@50    held-out P@20  held-out P@50
depth=2       0.800            0.400          0.440
depth=3       0.580            0.550          0.580


depth=4       0.780            0.550          0.580

hand rule on held-out clients:  Precision@20=0.300  Precision@50=0.400


**Does the gap hold?** No — not in the clean, monotonic way the in-sample numbers suggested.

On held-out clients, in this run: depth 2 lands around Precision@50 ≈ 0.44, and depth 3–4 land around ≈ 0.58 each — a real jump from depth 2 to 3, but then it *flattens*, not keeps climbing the way the in-sample curve did (0.64 → 0.66 → 0.72). The train-set number for depth 2 is also *higher* than its own held-out number (≈0.80 vs ≈0.44) — a classic sign the deeper/more-fit tree is partly memorizing patterns specific to the training clients, not signal that generalizes to a client it's never seen.

The hand rule drops too (Precision@50 ≈ 0.40 on held-out clients, down from 0.68 in-sample) — some of *its* apparent strength was also an artifact of scoring on the same data the rule was tuned against. But the tree still beats it at Precision@50 on new clients by a wide margin (≈0.58 vs ≈0.40), so the core lesson from Section 2 survives the honest test — it's just smaller and less tidy than the in-sample gap made it look.

**Takeaway:** in-sample numbers tell you what a model *memorized*; client-holdout numbers tell you what it *learned*. They can disagree, especially as you add depth. Always report the held-out number, and treat any in-sample metric as a rough sanity check at best — never the number that goes in front of a client.

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.